# Fire model: human review 1
Input audit, no training. The confirmed objective is to forecast
the start of new fires per region of Peru and selected future week.
The 10 km cells may be used internally, but the output will be regional.
This notebook presents artifacts of a full audit that was already run;
it does not need to download or load two million rows to review them again.

In [1]:
from pathlib import Path
import json
from IPython.display import display, Markdown

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data/series_metadata.json').is_file())
folder = repo / 'reports/prediction'
audit = json.loads((folder / 'input_audit.json').read_text(encoding='utf-8'))
contract = json.loads((folder / 'contract_proposed.json').read_text(encoding='utf-8'))
assert audit['model_trained'] is False
assert audit['public_api_snapshot_matches']
assert audit['our_snapshot_sha256_verified']
print('Estado:', contract['status'])
print('Commit fuente:', audit['source']['source_commit'])
print('Bytes del respaldo:', audit['source']['bytes'])
print('SHA-256:', audit['source']['sha256'])

Estado: experimental_dataset_ready_pending_training_review
Commit fuente: 3084cc5681bac763fdd651e01cef1d6a12461520
Bytes del respaldo: 131915596
SHA-256: 5382be98a0221e528995f2d44470d03f0e2bcc2307bc0a61410744c745aa7e71


## 1. What data we have
The COPY content of the dump was extracted with pg_restore in a container without network,
without running SQL or restoring into other projects' databases. The data are SP;
the teammate's source applies different filters from our series.

In [2]:
rows = ['| Tabla | Filas |', '| --- | ---: |']
rows += [f"| {name} | {item['rows']:,} |" for name, item in audit['tables'].items()]
display(Markdown('\n'.join(rows)))
print('Inicio:', audit['detections']['time_min'])
print('Fin:', audit['detections']['time_max'])
print('Detecciones por instrumento:', audit['detections']['categories']['instrument'])
print('Malla de nuestra API:', audit['our_series_metadata']['grid_km'], 'km')
print('Cobertura física corregida:', audit['our_series_metadata']['coverage_corrected'])

| Tabla | Filas |
| --- | ---: |
| meta | 10 |
| events | 795,540 |
| detections | 1,958,951 |
| static_sources | 4,803 |

Inicio: 2012-01-20 06:33:00+00:00
Fin: 2026-07-31 20:45:00+00:00
Detecciones por instrumento: {'VIIRS': 1762360, 'MODIS': 196591}
Malla de nuestra API: 10 km
Cobertura física corregida: False


## 2. Integrity is not ground truth
These checks detect structural problems. They do not prove that each
detection is an independent fire or that each absence is a day without fire.

In [3]:
checks = ['duplicate_ids', 'invalid_coordinates', 'invalid_timestamps',
          'invalid_or_negative_frp', 'missing_event_references']
for check in checks:
    print(check, audit['detections'][check])
    assert audit['detections'][check] == 0
assert sum(audit['detections']['missing_by_column'].values()) == 0
print('Columnas de detecciones sin faltantes en el respaldo auditado.')
print('Intervalos de eventos inválidos:', audit['events']['invalid_time_intervals'])

duplicate_ids 0
invalid_coordinates 0
invalid_timestamps 0
invalid_or_negative_frp 0
missing_event_references 0
Columnas de detecciones sin faltantes en el respaldo auditado.
Intervalos de eventos inválidos: 0


## 3. Future information we must avoid
The events were grouped retrospectively. Their duration, final centre,
classification and totals were not available at the start of the event.
The global static-source mask also needs a temporal reconstruction.

In [4]:
for number, finding in enumerate(audit['leakage_findings'], 1):
    print(f'{number}. {finding}')
display(Markdown(
    '**Decisión propuesta:** reconstruir variables desde detecciones anteriores '
    'al corte; no usar agregados finales, IDs retrospectivos ni máscaras futuras '
    'como si hubieran estado disponibles en tiempo real.'))

1. Event centroids, duration, totals, final type and confidence use detections after issuance.
2. Static-source masks in companion archive.py are fitted using the full 2012-2026 archive.
3. Event IDs and connected components are constructed retrospectively over the full archive.
4. date_local is solar longitude-based time; use t_utc and explicit America/Lima for operational dates.
5. Our full weekly estimates are only usable after the week ends, plus actual publication latency.
6. SP archives cannot reproduce historical NRT arrival times; operational equivalence is unverified.


**Decisión propuesta:** reconstruir variables desde detecciones anteriores al corte; no usar agregados finales, IDs retrospectivos ni máscaras futuras como si hubieran estado disponibles en tiempo real.

## 4. Proposed contract and next review
A row will be region/issue date/target week, labelled with at least
one new start during that week. A first record is an ignition proxy
that needs to be checked; ongoing events are not a new start.
The historical SP calendar does not prove
the operational availability of NRT; we need to record that latency.
The labels and the platform protocol will be frozen before fitting.

In [5]:
print('Horizonte:', contract['horizon_days'], 'días')
print('Anticipaciones solicitadas:', contract['forecast_lead_weeks'])
print('Emisión:', contract['issue_frequency'])
print('Candidatos:', ', '.join(contract['candidate_models']))
print('NRT preparado:', contract['NRT_ingestion_ready'])
print('Meteorología preparada:', contract['weather_ingestion_ready'])
print('Probabilidad física de ignición:', contract['physical_ignition_probability'])
display(Markdown(
    f"**Avance: {contract['stages_completed']} de 5 entregables, "
    f"{contract['milestone_progress_percent']} % por hitos.** "
    'No hay métricas predictivas nuevas ni modelo '
    'nuevo publicado. Validaremos periodos temporales y bloques espaciales, '
    'calibración y comparación con climatología antes de conectar el modelo.'))

Horizonte: 7 días
Anticipaciones solicitadas: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Emisión: weekly Sunday 00:00 America/Lima; first target starts next Monday
Candidatos: seasonal_climatology, simple_baseline, LightGBM
NRT preparado: True
Meteorología preparada: True
Probabilidad física de ignición: False


**Avance: 2 de 5 entregables, 45 % por hitos.** No hay métricas predictivas nuevas ni modelo nuevo publicado. Validaremos periodos temporales y bloques espaciales, calibración y comparación con climatología antes de conectar el modelo.

## 5. Pilot of causal labels, development 2013 only
Candidate rule: first episode detected after five days without a detection within
1.25 km. It is not ignition truth. The details of the cases need human review
before the training labels are approved.

In [6]:
pilot_path = folder / 'onset_pilot_report.json'
if pilot_path.is_file():
    pilot = json.loads(pilot_path.read_text(encoding='utf-8'))
    print('Detecciones de desarrollo:', pilot['rows_in_pilot'])
    print('Sensibilidad:', json.dumps(pilot['sensitivity'], indent=2))
    print('Casos de revisión:', pilot['human_review_cases'])
    print('Revisión humana terminada:', pilot['human_review_complete'])
    assert pilot['full_history_static_mask_used'] is False
    assert pilot['physical_ignition_ground_truth'] is False
    assert pilot['model_trained'] is False
    display(Markdown(
        '**Lectura:** el resultado regional de al menos un episodio es frecuente '
        'en este piloto. Necesitamos validar las etiquetas y la calibración por '
        'región; un porcentaje alto por sí solo no demuestra capacidad predictiva.'))
else:
    print('Piloto aún no ejecutado en esta copia del repositorio.')

Detecciones de desarrollo: 59225
Sensibilidad: [
  {
    "radius_m": 1250.0,
    "quiet_days": 5.0,
    "new_detected_episode_candidates": 33057,
    "regional_week_rows": 1300,
    "positive_regional_weeks": 1057,
    "regional_positive_fraction": 0.813076923076923
  },
  {
    "radius_m": 1000.0,
    "quiet_days": 3.0,
    "new_detected_episode_candidates": 35704,
    "regional_week_rows": 1300,
    "positive_regional_weeks": 1058,
    "regional_positive_fraction": 0.8138461538461539
  },
  {
    "radius_m": 2500.0,
    "quiet_days": 7.0,
    "new_detected_episode_candidates": 23895,
    "regional_week_rows": 1300,
    "positive_regional_weeks": 1051,
    "regional_positive_fraction": 0.8084615384615385
  }
]
Casos de revisión: 50
Revisión humana terminada: False


**Lectura:** el resultado regional de al menos un episodio es frecuente en este piloto. Necesitamos validar las etiquetas y la calibración por región; un porcentaje alto por sí solo no demuestra capacidad predictiva.